# D.R.O.N.A. — 06 · Gesture Policies with LeRobot (ACT / Diffusion)

A **clean, self-contained** LeRobot notebook that follows the *current official*
LeRobot docs (v0.6.x) — separate from the LLM notebook (04) so their dependencies
never clash.

**Key design choice:** D.R.O.N.A.'s gesture demonstrations are 6-DOF **joint
trajectories with no camera/video**, so we build a **state-only** `LeRobotDataset`
(`use_videos=False`). That deliberately avoids TorchCodec / PyAV / ffmpeg entirely —
the exact stack that caused the earlier failures.

Pipeline: **install → build dataset → train ACT → train Diffusion → evaluate**,
each step per the official docs:
<https://huggingface.co/docs/lerobot/en/installation> ·
<https://huggingface.co/docs/lerobot/en/act> ·
<https://huggingface.co/docs/lerobot/en/il_robots> ·
<https://huggingface.co/docs/lerobot/en/peft_training>

**Runtime:** Colab **GPU** (A100/T4). ACT trains in well under an hour.

## 0 · Setup — clone repo + install LeRobot (official way)

In [ ]:
# ===========================================================================
#  RUN THIS CELL FIRST.  Colab GPU runtime (Runtime > Change runtime type > GPU).
# ===========================================================================
import os, sys, subprocess, pathlib

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
      or "No GPU - set Runtime > Change runtime type > GPU")

REPO_URL = "https://github.com/trishan9/D.R.O.N.A.git"   # <-- EDIT if you forked

def _is_repo(p): return pathlib.Path(p, "drona", "__init__.py").is_file()
repo = next((p for p in (".", "/content/D.R.O.N.A") if _is_repo(p)), None)
if repo is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/D.R.O.N.A"], check=True)
    repo = "/content/D.R.O.N.A"
os.chdir(repo); print("repo:", os.getcwd())

# Official install: base lerobot + the `training` extra (docs/installation).
# ACT is in the base package; add [diffusion] later for the Diffusion policy.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[training]"], check=True)

# Optional: an HF token from Colab Secrets (key icon, name it HF_TOKEN) avoids
# anonymous rate limits when lerobot pulls anything.
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok: os.environ["HF_TOKEN"] = tok; print("HF token loaded from Colab Secrets")
except Exception:
    pass

import lerobot
print("lerobot version:", getattr(lerobot, "__version__", "?"))
print("setup complete")

## 1 · Get the gesture demonstrations

Upload **`drona_private_data.zip`** (it now contains
`data/demonstrations/demonstrations.jsonl` — 5,000 frames / 150 episodes across 6
gestures). The picker below **blocks** until you choose it.

In [ ]:
import glob, pathlib, subprocess
DEMOS = pathlib.Path("data/demonstrations/demonstrations.jsonl")
if not DEMOS.exists():
    found = (glob.glob("drona_private_data.zip") + glob.glob("/content/drona_private_data.zip")
             + glob.glob("/kaggle/input/**/drona_private_data.zip", recursive=True))
    if not found:
        try:
            from google.colab import files
            print(">>> Upload drona_private_data.zip (Choose Files) <<<")
            up = files.upload()
            found = [n for n in up if n.endswith(".zip")]
        except ImportError:
            pass
    if found:
        subprocess.run(["unzip", "-oq", found[0], "-d", "."], check=True)

assert DEMOS.exists(), "demonstrations.jsonl missing - upload drona_private_data.zip and re-run"
n = sum(1 for _ in open(DEMOS, encoding="utf-8"))
print(f"demonstrations ready: {n} frames -> {DEMOS}")

## 2 · Build a state-only LeRobotDataset (current API, no video)

Uses `LeRobotDataset.create(..., use_videos=False)` and `add_frame({..., 'task': ...})`
per the current API. No image features → no TorchCodec/PyAV.

In [ ]:
import json, collections, shutil, pathlib
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DOF, FPS, REPO_ID = 6, 20, "drona/gestures"
ROOT = pathlib.Path("data/lerobot/drona_gestures")
if ROOT.exists():
    shutil.rmtree(ROOT)   # LeRobotDataset.create wants a fresh dir

FEATURES = {
    "observation.state": {"dtype": "float32", "shape": (DOF,),
                          "names": [f"joint_{i}" for i in range(DOF)]},
    "action":            {"dtype": "float32", "shape": (DOF,),
                          "names": [f"joint_{i}" for i in range(DOF)]},
}
ds = LeRobotDataset.create(repo_id=REPO_ID, fps=FPS, features=FEATURES,
                           root=str(ROOT), use_videos=False)

# demonstrations.jsonl is FLAT (one record per frame); group by episode_index.
rows = [json.loads(l) for l in open("data/demonstrations/demonstrations.jsonl", encoding="utf-8")]
eps = collections.defaultdict(list)
for r in rows:
    eps[r["episode_index"]].append(r)

for ei in sorted(eps):
    for fr in sorted(eps[ei], key=lambda r: r["frame_index"]):
        ds.add_frame({
            "observation.state": np.asarray(fr["observation_state"], dtype=np.float32),
            "action":            np.asarray(fr["action"], dtype=np.float32),
            "task":              fr["gesture_label"],
        })
    ds.save_episode()

print(f"LeRobotDataset built: {len(eps)} episodes / {len(rows)} frames -> {ROOT}")

## 3 · Train ACT

Official `lerobot-train` CLI (docs/act). `steps=2000` is a fast smoke run — raise to
20000+ for the full model. Batch 8 per the ACT docs; drop to 4 on a small GPU.

In [ ]:
import subprocess, sys
ACT_STEPS = 2000     # raise to 20_000+ for the full model
BATCH     = 8        # ACT docs default; use 4 on a small GPU

cmd = [
    "lerobot-train",
    "--dataset.repo_id=drona/gestures",
    "--dataset.root=data/lerobot/drona_gestures",
    "--policy.type=act",
    "--policy.device=cuda",
    f"--batch_size={BATCH}",
    f"--steps={ACT_STEPS}",
    "--output_dir=outputs/train/act_drona",
    "--job_name=act_drona",
    "--wandb.enable=false",
]
print(">", " ".join(cmd))
r = subprocess.run(cmd, text=True, capture_output=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("--- STDERR (last 4000) ---\n" + (r.stderr or "")[-4000:])
    raise SystemExit("ACT training failed - see stderr above")
print("ACT training done -> outputs/train/act_drona")

## 4 · Train Diffusion Policy (optional ablation)

Needs the `diffusion` extra (docs/installation). Same CLI, `--policy.type=diffusion`.

In [ ]:
import subprocess, sys
TRAIN_DIFFUSION = True     # set False to skip
if TRAIN_DIFFUSION:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[diffusion]"], check=True)
    DIFF_STEPS, BATCH = 3000, 8
    cmd = [
        "lerobot-train",
        "--dataset.repo_id=drona/gestures",
        "--dataset.root=data/lerobot/drona_gestures",
        "--policy.type=diffusion",
        "--policy.device=cuda",
        f"--batch_size={BATCH}",
        f"--steps={DIFF_STEPS}",
        "--output_dir=outputs/train/diffusion_drona",
        "--job_name=diffusion_drona",
        "--wandb.enable=false",
    ]
    print(">", " ".join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("--- STDERR (last 4000) ---\n" + (r.stderr or "")[-4000:])
        raise SystemExit("Diffusion training failed - see stderr above")
    print("Diffusion training done -> outputs/train/diffusion_drona")
else:
    print("skipped")

## 5 · Evaluate — open-loop trajectory reproduction (C3)

There is no camera or sim env for these gestures, so the official `lerobot-rollout`
(real-robot) path does not apply. Instead we load the trained policy and measure how
well it **reproduces the demonstrated actions** (MSE) and how **smooth** they are
(mean |jerk| — the C3 metric), on held-out episodes.

In [ ]:
import json, collections, glob, pathlib
import numpy as np, torch
from lerobot.policies.act.modeling_act import ACTPolicy

def _latest_ckpt(base):
    cands = sorted(glob.glob(f"{base}/checkpoints/*/pretrained_model"))
    return cands[-1] if cands else None

def mean_jerk(actions):           # actions: (T, DOF)
    a = np.asarray(actions, np.float32)
    return float(np.mean(np.abs(np.diff(a, n=3, axis=0)))) if len(a) > 3 else float("nan")

# held-out: last episode of each gesture
rows = [json.loads(l) for l in open("data/demonstrations/demonstrations.jsonl", encoding="utf-8")]
eps = collections.defaultdict(list)
for r in rows:
    eps[r["episode_index"]].append(r)
by_gesture = collections.defaultdict(list)
for ei, frs in eps.items():
    by_gesture[frs[0]["gesture_label"]].append(ei)
heldout = [max(v) for v in by_gesture.values()]

ck = _latest_ckpt("outputs/train/act_drona")
print("ACT checkpoint:", ck)
try:
    policy = ACTPolicy.from_pretrained(ck); policy.eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; policy.to(dev)
    mses, jerks_pred, jerks_gt = [], [], []
    for ei in heldout:
        frs = sorted(eps[ei], key=lambda r: r["frame_index"])
        gt = np.asarray([f["action"] for f in frs], np.float32)
        preds = []
        policy.reset()
        for f in frs:
            obs = {"observation.state": torch.tensor(f["observation_state"], dtype=torch.float32,
                                                     device=dev).unsqueeze(0)}
            with torch.no_grad():
                a = policy.select_action(obs)
            preds.append(a.squeeze(0).cpu().numpy())
        preds = np.asarray(preds, np.float32)
        mses.append(float(np.mean((preds - gt) ** 2)))
        jerks_pred.append(mean_jerk(preds)); jerks_gt.append(mean_jerk(gt))
    print(f"\nACT open-loop eval on {len(heldout)} held-out episodes:")
    print(f"  action MSE      : {np.mean(mses):.4f}")
    print(f"  mean|jerk| pred : {np.nanmean(jerks_pred):.2f}  (demo GT: {np.nanmean(jerks_gt):.2f})")
    json.dump({"n_episodes": len(heldout), "action_mse": float(np.mean(mses)),
               "jerk_pred": float(np.nanmean(jerks_pred)), "jerk_gt": float(np.nanmean(jerks_gt))},
              open("outputs/act_eval.json", "w"), indent=2)
    print("saved -> outputs/act_eval.json")
except Exception as exc:
    print(f"eval skipped ({type(exc).__name__}: {str(exc)[:160]})")
    print("The checkpoints are still trained + saved; this open-loop eval is best-effort.")

## 6 · Save checkpoints to Google Drive

In [ ]:
import shutil, pathlib
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = pathlib.Path("/content/drive/MyDrive/drona_training/lerobot")
    dst.mkdir(parents=True, exist_ok=True)
    for src in ("outputs/train/act_drona", "outputs/train/diffusion_drona", "outputs/act_eval.json"):
        s = pathlib.Path(src)
        if s.exists():
            d = dst / s.name
            shutil.copytree(s, d, dirs_exist_ok=True) if s.is_dir() else shutil.copy2(s, d)
            print("saved ->", d)
    print("\nLeRobot policies persisted to Drive/drona_training/lerobot")
except Exception as exc:
    print("Drive not available:", exc, "- download outputs/ manually before the session ends")